# Homework 02: Visual Search with Pretrained and Fine-Tuned Embeddings

**Student Version**

**Release:** May 28, 2026  
**Deadline:** May 31, 2026  
**Recommended time budget:** 4-6 focused hours  
**Recommended environment:** Google Colab with GPU for fine-tuning. The frozen retrieval path can run on CPU, but it will be slower.

In this homework you will build an image-retrieval system, evaluate it, and then try to improve it. The central idea is that a neural network can map images into an embedding space where similar images are close to each other.

Goals for this homework:
- Build a **visual search** pipeline with image embeddings
- Use a pretrained **ResNet18** backbone as an embedding extractor
- Retrieve similar images with **cosine similarity**
- Evaluate the system with **Recall@K** and qualitative examples
- Improve the embeddings by **fine-tuning part of the backbone**
- Compare the frozen and fine-tuned systems in a fair way

Minimum viable submission:
- frozen pretrained embedding system
- gallery/query embeddings extracted with the same model
- at least three retrieval visualizations
- `Recall@1`, `Recall@5`, and same-class vs different-class similarity analysis
- short interpretation of retrieval successes and failures

Full-credit submission also includes:
- a short fine-tuning experiment
- a fair frozen vs fine-tuned comparison using the same gallery/query split
- a written explanation of whether fine-tuning helped

Grading guide:

| Part | Weight | What matters most |
|---|---:|---|
| Task 1: frozen embedding setup | 20% | correct preprocessing, splits, loaders, frozen ResNet18 embedding model |
| Task 2: retrieval pipeline | 20% | normalized embeddings, cosine search, clear retrieval visualizations |
| Task 3: retrieval evaluation | 25% | Recall@1/5, similarity distributions, interpretation |
| Task 4: fine-tuning improvement | 20% | fair setup, partial unfreezing, short training run, rerun retrieval |
| Task 5: comparison and conclusions | 15% | comparison table, side-by-side examples, honest analysis |

Do not chase state-of-the-art CIFAR-10 retrieval. The goal is a reproducible retrieval experiment and a fair comparison.


## 0. Setup

The helper code below is provided so that the homework focuses on retrieval ideas and comparison rather than setup boilerplate.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class TransformedSubset(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def make_cifar_splits(transform, train_size=8000, val_size=2000, gallery_size=2000, query_size=300, seed=42):
    train_base = datasets.CIFAR10(root='data', train=True, download=True, transform=None)
    test_base = datasets.CIFAR10(root='data', train=False, download=True, transform=None)

    rng = random.Random(seed)
    all_train_indices = list(range(len(train_base)))
    all_test_indices = list(range(len(test_base)))
    rng.shuffle(all_train_indices)
    rng.shuffle(all_test_indices)

    train_indices = all_train_indices[:train_size]
    val_indices = all_train_indices[train_size:train_size + val_size]
    gallery_indices = all_test_indices[:gallery_size]
    query_indices = all_test_indices[gallery_size:gallery_size + query_size]

    train_ds = TransformedSubset(train_base, train_indices, transform=transform)
    val_ds = TransformedSubset(train_base, val_indices, transform=transform)
    gallery_ds = TransformedSubset(test_base, gallery_indices, transform=transform)
    query_ds = TransformedSubset(test_base, query_indices, transform=transform)
    return train_ds, val_ds, gallery_ds, query_ds, train_base.classes


def make_loader(dataset, batch_size=128, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)


def show_images(dataset, class_names, indices=None, max_items=8, title=None):
    if indices is None:
        indices = list(range(min(max_items, len(dataset))))
    fig, axes = plt.subplots(1, len(indices), figsize=(2.2 * len(indices), 2.8))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices):
        image, label = dataset[idx]
        image_np = image.permute(1, 2, 0).cpu().numpy()
        image_np = (image_np - image_np.min()) / max(1e-6, image_np.max() - image_np.min())
        ax.imshow(image_np)
        ax.set_title(class_names[label])
        ax.axis('off')
    if title is not None:
        plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def make_embedding_model():
    raise NotImplementedError('Create a pretrained embedding extractor in Task 1')


def extract_embeddings(model, loader):
    model.eval()
    all_embeddings, all_labels, all_images = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            embeddings = model(images.to(device))
            norms = embeddings.norm(dim=1, keepdim=True)
            embeddings = embeddings / norms
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels.cpu())
            all_images.append(images.cpu())
    return torch.cat(all_embeddings), torch.cat(all_labels), torch.cat(all_images)



def search_similar_images(query_embeddings, gallery_embeddings, top_k=5):
    # TODO in Task 2:
    # 1. compute cosine similarity scores between each query and each gallery image
    # 2. choose the top-k gallery images for every query
    # Hint: embeddings are already normalized by extract_embeddings(...), so dot product is cosine similarity.
    # A simple solution can use:
    # scores = query_embeddings @ gallery_embeddings.T
    # all_top_scores = []
    # all_top_indices = []
    # for query_scores in scores:
    #     sorted_indices = torch.argsort(query_scores, descending=True)
    #     query_top_indices = sorted_indices[:top_k]
    #     query_top_scores = query_scores[query_top_indices]
    #     all_top_scores.append(query_top_scores)
    #     all_top_indices.append(query_top_indices)
    # return torch.stack(all_top_scores), torch.stack(all_top_indices)
    raise NotImplementedError('Implement search_similar_images in Task 2')


def recall_at_k(query_labels, gallery_labels, top_indices, k):
    # TODO in Task 3:
    # For each query, look at the first k retrieved gallery indices.
    # Count it as a hit if at least one retrieved label matches the query label.
    # Return hits / number of queries as a Python float.
    # A simple loop is completely fine here.
    raise NotImplementedError('Implement recall_at_k in Task 3')


def plot_retrieval(query_images, query_labels, gallery_images, gallery_labels, top_indices, class_names, query_idx=0):
    retrieved = top_indices[query_idx].tolist()
    total = 1 + len(retrieved)
    fig, axes = plt.subplots(1, total, figsize=(2.4 * total, 3.2))

    q_image = query_images[query_idx].permute(1, 2, 0).numpy()
    q_image = (q_image - q_image.min()) / max(1e-6, q_image.max() - q_image.min())
    axes[0].imshow(q_image)
    axes[0].set_title(f'Query\n{class_names[query_labels[query_idx]]}')
    axes[0].axis('off')

    for ax, idx in zip(axes[1:], retrieved):
        image = gallery_images[idx].permute(1, 2, 0).numpy()
        image = (image - image.min()) / max(1e-6, image.max() - image.min())
        ax.imshow(image)
        ax.set_title(class_names[gallery_labels[idx]])
        ax.axis('off')
    plt.tight_layout()
    plt.show()



def sample_pair_similarities(embeddings, labels, pairs_per_type=300, seed=42):
    rng = random.Random(seed)

    indices_by_label = {}
    for index, label in enumerate(labels.tolist()):
        if label not in indices_by_label:
            indices_by_label[label] = []
        indices_by_label[label].append(index)

    class_labels = list(indices_by_label.keys())
    same_scores = []
    diff_scores = []

    for _ in range(pairs_per_type):
        label = rng.choice(class_labels)
        first_index, second_index = rng.sample(indices_by_label[label], 2)
        same_score = embeddings[first_index] @ embeddings[second_index]
        same_scores.append(float(same_score))

        first_label, second_label = rng.sample(class_labels, 2)
        first_index = rng.choice(indices_by_label[first_label])
        second_index = rng.choice(indices_by_label[second_label])
        diff_score = embeddings[first_index] @ embeddings[second_index]
        diff_scores.append(float(diff_score))

    return same_scores, diff_scores


def train_classifier(model, train_loader, val_loader, epochs, lr, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    trainable_parameters = []
    for parameter in model.parameters():
        if parameter.requires_grad:
            trainable_parameters.append(parameter)
    optimizer = torch.optim.AdamW(trainable_parameters, lr=lr, weight_decay=weight_decay)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            train_total += labels.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                val_loss += loss.item() * labels.size(0)
                val_correct += (logits.argmax(dim=1) == labels).sum().item()
                val_total += labels.size(0)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss / train_total,
            'train_acc': train_correct / train_total,
            'val_loss': val_loss / val_total,
            'val_acc': val_correct / val_total,
        })
        print(
            f"Epoch {epoch:02d} | train_acc={history[-1]['train_acc']:.4f} | val_acc={history[-1]['val_acc']:.4f}"
        )
    return pd.DataFrame(history)


def plot_history(history_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val loss')
    axes[0].set_title(f'{title}: loss')
    axes[0].legend()
    axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train acc')
    axes[1].plot(history_df['epoch'], history_df['val_acc'], label='val acc')
    axes[1].set_title(f'{title}: accuracy')
    axes[1].legend()
    plt.tight_layout()
    plt.show()


set_seed(42)


## Helper Function Contracts

The setup cell above defines helper functions so you can focus on retrieval and comparison rather than boilerplate. Most helpers are provided. Two short algorithmic helpers, `search_similar_images(...)` and `recall_at_k(...)`, are intentionally left for you to implement.

| Helper | Use it when | Inputs | Returns |
|---|---|---|---|
| `make_cifar_splits(transform, train_size=8000, val_size=2000, gallery_size=2000, query_size=300, seed=42)` | You need train/validation/gallery/query datasets | preprocessing transform, split sizes, seed | `train_ds, val_ds, gallery_ds, query_ds, class_names` |
| `make_loader(dataset, batch_size=128, shuffle=False)` | You need a PyTorch DataLoader | dataset, batch size, shuffle flag | `DataLoader` |
| `show_images(dataset, class_names, indices=None, max_items=8, title=None)` | You want to inspect dataset examples | dataset, class names, optional indices/count/title | displays a figure, returns nothing |
| `extract_embeddings(model, loader)` | You need normalized embeddings for a loader | embedding model, loader | `embeddings, labels, images` tensors on CPU |
| `search_similar_images(query_embeddings, gallery_embeddings, top_k=5)` | You need nearest neighbors; **you implement this** | query embeddings, gallery embeddings, number of results | `top_scores, top_indices` |
| `recall_at_k(query_labels, gallery_labels, top_indices, k)` | You need retrieval accuracy; **you implement this** | query labels, gallery labels, retrieval indices, `k` | float recall value |
| `plot_retrieval(query_images, query_labels, gallery_images, gallery_labels, top_indices, class_names, query_idx=0)` | You need to visualize one query and its retrieved images | query/gallery images and labels, top indices, class names, query index | displays a figure, returns nothing |
| `sample_pair_similarities(embeddings, labels, pairs_per_type=300, seed=42)` | You need same-class vs different-class cosine similarities | embeddings, labels, number of sampled pairs, seed | `same_scores, diff_scores` lists |
| `train_classifier(model, train_loader, val_loader, epochs, lr, weight_decay=0.0)` | You need to fine-tune a classifier | model, train/val loaders, epochs, learning rate, weight decay | pandas `DataFrame` with train/validation loss and accuracy |
| `plot_history(history_df, title)` | You need fine-tuning curves | history DataFrame from `train_classifier`, plot title | displays loss and accuracy curves, returns nothing |

`make_embedding_model()` is intentionally incomplete in the setup cell. In Task 1, either implement that helper or create `frozen_embedding_model` directly in the task cell.

For `search_similar_images(...)` and `recall_at_k(...)`, fill in the helper definitions in the setup cell before using them in Tasks 2 and 3. Prefer the simple versions shown in the comments over clever one-liners.


## 1. Build the frozen embedding system

### Exercise 1.1
Load pretrained `ResNet18` weights and create an embedding extractor that outputs feature vectors instead of class logits.

Use:
- `weights = ResNet18_Weights.DEFAULT`
- `preprocess = weights.transforms()`
- `models.resnet18(weights=weights)`
- `nn.Identity()` to replace the classification head

### Exercise 1.2
Use `make_cifar_splits(...)` with the official preprocessing transform and create:
- a training subset for later fine-tuning
- a validation subset
- a gallery subset
- a query subset

Then create loaders with `make_loader(...)` and visualize a few gallery/query examples with `show_images(...)`.

Recommended split sizes for the full run:
- `train_size=8000`
- `val_size=2000`
- `gallery_size=2000`
- `query_size=300`

If your machine is slow, you may temporarily debug with smaller sizes, but use the recommended sizes for your final reported results if possible.


In [ ]:
# Exercise 1

# TODO:
# 1. load pretrained ResNet18 weights
# 2. get the official transform from weights.transforms()
# 3. build train / val / gallery / query datasets with make_cifar_splits(...)
# 4. create train / val / gallery / query loaders with make_loader(...)
# 5. visualize gallery and query examples with show_images(...)
# 6. build a frozen embedding model by replacing the ResNet18 `fc` layer with nn.Identity()

# Suggested variables:
# weights
# preprocess
# train_ds, val_ds, gallery_ds, query_ds, class_names
# train_loader, val_loader, gallery_loader, query_loader
# frozen_embedding_model

raise NotImplementedError('Implement Exercise 1')


In [ ]:
# Task 1 sanity check
# Run this after you finish Exercise 1.

assert len(class_names) == 10
assert len(gallery_ds) > 0 and len(query_ds) > 0
assert next(frozen_embedding_model.parameters()).device == device
xb, yb = next(iter(gallery_loader))
with torch.no_grad():
    test_emb = frozen_embedding_model(xb[:4].to(device))
assert test_emb.ndim == 2, 'Embedding model should return a 2D tensor: batch x embedding_dim'
print('Embedding shape:', tuple(test_emb.shape))
print('Classes:', class_names)


## 2. Extract embeddings and run retrieval

### Task 2.1
Use `extract_embeddings(...)` to extract normalized embeddings for the gallery and query images.

### Task 2.2
First implement `search_similar_images(...)` in the setup cell. A simple version should compute a score matrix, sort each row, and keep the first `top_k` columns. Then use it with `top_k=5` and visualize at least three query examples with `plot_retrieval(...)`.

The gallery is the searchable database. The query set contains the images you ask the system to find neighbors for.


In [ ]:
# Task 2

# TODO:
# 1. extract gallery embeddings, labels, images with extract_embeddings(...)
# 2. extract query embeddings, labels, images with extract_embeddings(...)
# 3. implement search_similar_images(...) in the setup cell if you have not yet done so
# 4. run search_similar_images(..., top_k=5)
# 5. visualize at least three retrieval examples with plot_retrieval(...)

# Suggested variables:
# frozen_gallery_emb, frozen_gallery_labels, frozen_gallery_images
# frozen_query_emb, frozen_query_labels, frozen_query_images
# frozen_scores, frozen_top_indices

raise NotImplementedError('Implement Task 2')


## 3. Evaluate the frozen system

Evaluate retrieval quality, not just pretty pictures.

Required:
- implement `recall_at_k(...)` in the setup cell
- `Recall@1` using `recall_at_k(..., k=1)`
- `Recall@5` using `recall_at_k(..., k=5)`
- same-class vs different-class cosine similarity using `sample_pair_similarities(...)`
- one short written interpretation

Recall@K answers this question: for each query image, did at least one image from the same class appear in the top K retrieved results? A readable loop over queries is perfectly acceptable.


In [ ]:
# Task 3

# TODO:
# 1. implement recall_at_k(...) in the setup cell if you have not yet done so
# 2. compute Recall@1 and Recall@5 for the frozen system with recall_at_k(...)
# 3. sample same-class and different-class similarity scores with sample_pair_similarities(...)
# 4. visualize the two distributions with a histogram or box plot

# Suggested variables:
# frozen_recall1, frozen_recall5
# frozen_same_scores, frozen_diff_scores

raise NotImplementedError('Implement Task 3')


In [ ]:
# Task 3 sanity check
# Run this after you finish Task 3.

assert 0.0 <= frozen_recall1 <= 1.0
assert 0.0 <= frozen_recall5 <= 1.0
assert frozen_recall5 >= frozen_recall1, 'Recall@5 should be at least Recall@1'
assert len(frozen_same_scores) > 0 and len(frozen_diff_scores) > 0
print('Frozen Recall@1:', round(float(frozen_recall1), 4))
print('Frozen Recall@5:', round(float(frozen_recall5), 4))
print('Mean same-class similarity:', round(float(np.mean(frozen_same_scores)), 4))
print('Mean different-class similarity:', round(float(np.mean(frozen_diff_scores)), 4))


### Task 3 written interpretation

Write 3-5 sentences here:

- How good is the frozen retrieval system according to Recall@1 and Recall@5?
- Are same-class similarities usually higher than different-class similarities?
- Which qualitative examples looked good or bad?


## 4. Improve the system with fine-tuning

Now improve the embeddings by fine-tuning part of the pretrained backbone on CIFAR-10 classification.

Suggested strategy:
- start from pretrained `ResNet18`
- replace `fc` with a 10-class CIFAR-10 classifier
- freeze most parameters
- unfreeze only `layer4` and `fc`
- train for a few epochs with `train_classifier(...)`
- after fine-tuning, copy the model, replace `fc` with `nn.Identity()`, and reuse it as an embedding extractor

Important: use the **same gallery/query split** as before so the comparison stays fair. Fine-tuning may or may not improve every query. The important part is that your comparison is controlled and honestly interpreted.

Runtime note: if fine-tuning is slow, train for 2 epochs while debugging, then use 3-4 epochs for the final run if possible.


In [ ]:
# Task 4

# TODO:
# 1. create a classification model initialized from pretrained ResNet18
# 2. replace the final fc layer with nn.Linear(num_features, len(class_names))
# 3. freeze most layers and unfreeze layer4 + fc
# 4. fine-tune for a few epochs on the training subset with train_classifier(...)
# 5. plot fine-tuning curves with plot_history(...)
# 6. convert the model back into an embedding extractor by replacing fc with nn.Identity()
# 7. extract fine-tuned gallery/query embeddings with extract_embeddings(...)
# 8. rerun retrieval and evaluation with search_similar_images(...) and recall_at_k(...)

# Suggested variables:
# finetune_model
# finetune_history
# finetuned_embedding_model
# tuned_gallery_emb, tuned_gallery_labels, tuned_gallery_images
# tuned_query_emb, tuned_query_labels, tuned_query_images
# tuned_scores, tuned_top_indices
# tuned_recall1, tuned_recall5

raise NotImplementedError('Implement Task 4')


## 5. Compare frozen vs fine-tuned retrieval

This is the most important part of the homework.

Required:
- one comparison table for frozen vs fine-tuned embeddings
- at least three query examples where you compare the retrieval results side by side
- a short discussion of whether fine-tuning helped and where it still failed

A good comparison does not only say "fine-tuning is better" or "fine-tuning is worse." It points to metrics and examples.


In [ ]:
# Task 5

# TODO:
# 1. create a comparison DataFrame with Recall@1 and Recall@5
# 2. compare several query examples side by side with plot_retrieval(...)
# 3. write short conclusions below the code cell

raise NotImplementedError('Implement Task 5')


### Task 5 written conclusion

Write 4-6 sentences here:

- Did fine-tuning improve Recall@1 or Recall@5?
- Did the side-by-side examples match the metrics?
- Which classes remained difficult?
- Was the comparison fair? Mention the same gallery/query split.


## 6. Homework deliverable

Before submitting, check that your final notebook contains:
- [ ] a frozen pretrained retrieval system
- [ ] retrieval visualizations for at least three queries
- [ ] `Recall@1` and `Recall@5`
- [ ] same-class vs different-class similarity analysis
- [ ] one fine-tuned retrieval system
- [ ] a frozen vs fine-tuned comparison table
- [ ] side-by-side qualitative comparison examples
- [ ] short written conclusions
- [ ] no unresolved `NotImplementedError` cells in the required path

## 7. Wrap-up questions

Please answer briefly in markdown:

1. Why do embeddings make sense for retrieval tasks?
2. Why is cosine similarity a reasonable default here?
3. Did fine-tuning improve retrieval quality? How do you know?
4. Which classes or query types remained difficult?
5. If you had one more day, what would you try next?

## Optional extension

Try one extra idea, for example:
- use a different pretrained backbone
- change the train subset size
- visualize nearest neighbors for failure cases only
- try a tiny Siamese / triplet-loss experiment on a small subset
